In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import logging
from tqdm import tqdm

from eq_mag_prediction.utilities import data_utils
from etas.rate_computation import (
    rate_at_t_all_grid,
    return_local_rate,  # Optional: for single-point rate computation
    mu, kappa, g, f,
    xp,  # Unified array module (CuPy or NumPy)
    set_use_gpu,
)

# GPU configuration: set to False to force CPU (e.g. for debugging or limited GPU memory)
USE_GPU = True
set_use_gpu(USE_GPU)

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


In [ ]:
logger.info("Loading catalog data...")
catalog = data_utils.hauksson_dataframe()
logger.info(f"Loaded catalog with {len(catalog)} events")
catalog


In [ ]:
plt.plot(catalog.time, catalog.magnitude)


In [ ]:
plt.hist(np.diff(catalog.time.values[20:-100]), np.logspace(0, 9, 100))
plt.xscale('log')


In [ ]:
# TODO: "mu" should be a function of space
params = {
    "mu": 0.1,
    "beta": np.log(10),
    "alpha": 1,
    "gamma": 1.3,
    "c": -2.2,
    "p": 1,
    "D": 1,
    "q": 1,
    "m0": 1,
}

# Note: ETAS model functions (mu, kappa, g, f) are now imported from etas.rate_computation
# They take params as an argument, e.g., mu(x, y, params), kappa(m, params), etc.


In [ ]:
start_forecast = 1.2e9
end_forecast = 1.2005e9
dt = 1

time_vec = np.arange(start_forecast, end_forecast, dt)

history = catalog[catalog.time < start_forecast]
logger.info(f"Forecast period: {start_forecast:.2e} to {end_forecast:.2e}")
logger.info(f"History contains {len(history)} events before forecast start")


In [ ]:
# Spatial boundaries
x_min, x_max = history.x_utm.min(), history.x_utm.max()
y_min, y_max = history.y_utm.min(), history.y_utm.max()

# Grid size
dx = 1
dy = 1

# Grid
x_grid = np.linspace(x_min, x_max, 4)
y_grid = np.linspace(y_min, y_max, 4)
XX, YY = np.meshgrid(x_grid, y_grid)
logger.info(f"Spatial grid: {XX.shape[0]}x{XX.shape[1]} = {XX.size} grid points")
logger.info(f"X range: [{x_min:.2f}, {x_max:.2f}], Y range: [{y_min:.2f}, {y_max:.2f}]")


In [ ]:
XX.shape


In [ ]:
def magnitude_gen(n, m0=params["m0"], beta=params["beta"]):
    np.random.seed(42)
    return np.random.exponential(scale=1/beta, size=n) + m0


In [ ]:
def get_time_step(
    rate_function: callable,
    current_time: float,
    ):
    B = rate_function(current_time)
    dt =  np.random.exponential(scale=1/B)
    future_rate = rate_function(current_time + dt)
    U = np.random.uniform(0,1)
    reject = U >= future_rate/B
    return (dt, reject)

def propagate_catalog(
    rate_function: callable,
    current_time: float,
    catalog: pd.DataFrame,
    x_utm: float,
    y_utm: float,
    ):
    dt, reject = get_time_step(rate_function, current_time)
    if not reject:
        catalog.append({'time':current_time + dt, 'magnitude':magnitude_gen(1), 'x_utm':x_utm, 'y_utm':y_utm}, ignore_index=True)
    return current_time + dt


In [ ]:
# def rate_at_t_all_grid(t_days, x_flat, y_flat, h_x, h_y, h_m, h_t_days):
#     """
#     Compute ETAS intensity at time(s) for all grid points.
    
#     Args:
#         t_days: Current time (in days) - scalar
#         x_flat: 1D array of x coordinates for all grid points
#         y_flat: 1D array of y coordinates for all grid points
#         h_x: 1D array of x coordinates from history
#         h_y: 1D array of y coordinates from history
#         h_m: 1D array of magnitudes from history
#         h_t: 1D array of times from history (in days)
    
#     Returns:
#         1D array of intensities at each grid point
#     """
#     n_grid = len(x_flat)
#     local_intensity = np.zeros(n_grid)
    
#     # Loop over events (vectorized operations inside)
#     for k in range(len(h_x)):
#         x_k, y_k, m_k, t_k_days = h_x[k], h_y[k], h_m[k], h_t_days[k]
        
#         # Vectorized computation for all grid points
#         dx = x_flat - x_k
#         dy = y_flat - y_k
#         kappa_val = kappa(m_k)
#         f_vals = f(dx, dy, m_k)
#         # g() handles scalar or array input - numpy broadcasting handles it
#         g_vals = g(t_days - t_k_days)
        
#         local_intensity += kappa_val * f_vals * g_vals
    
#     # Add background rate (mu is constant)
#     mu_val = mu(x_flat[0], y_flat[0])  # mu doesn't depend on x,y
#     return mu_val + local_intensity

# Note: return_local_rate() and rate_at_t_all_grid() are now imported from etas.rate_computation
# They support GPU acceleration and take params as an argument:
#   - rate_at_t_all_grid(t_days, x_flat, y_flat, h_x, h_y, h_m, h_t_days, params)
#   - return_local_rate(history, x, y, params)
# The GPU-accelerated rate_at_t_all_grid() is used in the main simulation loop.


In [ ]:
# estimate the rate for each pixel at each time step
# Precompute grid arrays once (optimization)
x_flat = XX.flatten()
y_flat = YY.flatten()
n_grid = x_flat.size



logger.info("Starting ETAS simulation...")
forecast_cat = pd.DataFrame({'time':[None], 'magnitude':[None], 'x_utm':[None], 'y_utm':[None]})
t = start_forecast
total_time_steps = (end_forecast - start_forecast) / dt
events_generated = 0
iterations = 0

# Create progress bar for main loop
pbar = tqdm(total=end_forecast - start_forecast, unit='time', desc='Simulation progress', 
            initial=0, bar_format='{l_bar}{bar}| {n:.2e}/{total:.2e} [{elapsed}<{remaining}]')

# for i,t in enumerate(time_vec):
while t < end_forecast:
    iterations += 1
    # time_step_events = {'time':[], 'magnitude':[], 'x_utm':[], 'y_utm':[]}
    # # iterate upon gridpoints
    # summed_intensity = np.empty_like(XX)
    # dt_vec = []
    # reject_vec = []
    
    # # Progress bar for grid point iteration
    # grid_points = list(zip(XX.flatten(), YY.flatten()))
    # for x, y in tqdm(grid_points, desc=f'  Grid points (iter {iterations})', 
    #                  leave=False, disable=len(grid_points) < 10):
    #     current_local_rate = return_local_rate(history, x, y)
    #     dt, reject = get_time_step(current_local_rate, t)   # dt is in days
    #     dt_vec.append(dt)
    #     reject_vec.append(reject)
    

        # Extract history to numpy arrays at start of iteration (ensures latest history is used)
    h_x = history['x_utm'].values
    h_y = history['y_utm'].values
    h_m = history['magnitude'].values
    h_t_days = history['time'].values / 86400.0
    
    # Vectorized rate computation at current time t for all grid points (GPU-accelerated)
    B = rate_at_t_all_grid(t/86400.0, x_flat, y_flat, h_x, h_y, h_m, h_t_days, params)
    
    # Convert B to GPU array if using GPU (rate_at_t_all_grid returns NumPy for pandas compatibility)
    # Check if xp is cupy by checking if it has cuda module
    if hasattr(xp, 'cuda'):
        B = xp.asarray(B)
    
    # Vectorized thinning step: sample dt for each grid point
    # Avoid division by zero by clipping B to a small positive value
    # Use xp for GPU-accelerated random number generation
    dt_vec = xp.random.exponential(1 / xp.clip(B, 1e-15, None))  # dt_vec is in days
    
    # Vectorized rate computation at future times t+dt_vec for all grid points
    rate_future = rate_at_t_all_grid(t/86400.0 + dt_vec, x_flat, y_flat, h_x, h_y, h_m, h_t_days, params)
    
    # Convert rate_future to GPU array if using GPU
    if hasattr(xp, 'cuda'):
        rate_future = xp.asarray(rate_future)
    
    # Vectorized rejection sampling (use xp for GPU acceleration)
    U = xp.random.uniform(size=n_grid)
    reject_vec = U >= (rate_future / B)
    
    # Convert arrays back to numpy for pandas operations if needed
    if hasattr(dt_vec, 'get'):  # GPU array
        dt_vec = dt_vec.get()
    if hasattr(reject_vec, 'get'):  # GPU array
        reject_vec = reject_vec.get()


    # dt_vec = np.array(dt_vec)
    # reject_vec = np.array(reject_vec)
    # Find the smallest dt where reject_vec is False
    valid_dt = dt_vec[~reject_vec]
    min_dt = valid_dt.min() if valid_dt.size > 0 else None

    if min_dt is None:
        # All proposals rejected: use random choice
        # Convert to numpy array if needed for random.choice
        dt_vec_np = np.asarray(dt_vec)
        min_dt = np.random.choice(dt_vec_np)
        logger.debug(f"Iteration {iterations}: No valid dt found, using random choice: {min_dt:.2e} days")
    else:
        # x = XX.flatten()[np.argmin(dt_vec)]
        # y = YY.flatten()[np.argmin(dt_vec)]
        # Find winner index: smallest accepted dt (FIX: use valid indices, not global argmin)
        valid_idx = np.flatnonzero(~reject_vec)
        winner = valid_idx[np.argmin(dt_vec[valid_idx])]
        
        x = x_flat[winner]
        y = y_flat[winner]

        mag_array = magnitude_gen(1)
        mag_value = mag_array.item() if isinstance(mag_array, np.ndarray) else mag_array
        # catalog_append = {'time':t+min_dt, 'magnitude':mag_value, 'x_utm':x, 'y_utm':y}
        lon, lat = data_utils.PROJECTIONS['california'](x, y, inverse=True)
        catalog_append = {'time':t+(min_dt * 86400), 'magnitude':mag_value, 'x_utm':x, 'y_utm':y, 'longitude':lon, 'latitude':lat}

        history.loc[len(history)] = catalog_append
        events_generated += 1
        logger.debug(f"Iteration {iterations}: Generated event at t={t+min_dt * 86400:.2e}, m={mag_value:.2f}")
    
    # t += min_dt * 86400  # Convert min_dt to seconds
    # pbar.update(min_dt * 86400)
    
    # Log progress every 100 iterations
    if iterations % 100 == 0:
        logger.info(f"Iteration {iterations}: t={t:.2e}, events={events_generated}, history_size={len(history)}")
        # Update time (convert min_dt from days to seconds)
    t += min_dt * 86400
    
    # Update progress bar (guard against None/invalid min_dt)
    if min_dt is not None and np.isfinite(min_dt) and min_dt > 0:
        pbar.update(min_dt * 86400)


pbar.close()
logger.info(f"Simulation complete: {iterations} iterations, {events_generated} events generated")
logger.info(f"Final catalog size: {len(history)} events")


In [ ]:
U >= (rate_future / B)


In [ ]:
np.array([2]*len(B)) >= np.array(rate_future / B)


In [ ]:
(rate_future / B)


In [ ]:
B


In [ ]:
rate_future


In [ ]:
events_generated


In [ ]:
t


In [ ]:
history


In [ ]:
for index, row in history.iterrows():
    # if pd.isna(row.longitude) or pd.isna(row.latitude):
    lon, lat = data_utils.PROJECTIONS['california'](row.x_utm, row.y_utm, inverse=True)
    history.loc[index, 'longitude'] = lon
    history.loc[index, 'latitude'] = lat

history 


In [ ]:
start_forecast


In [ ]:
from eq_mag_prediction.utilities


In [ ]:
histor
